# 07 — Whisper LoRA Fine-Tuning

Dans ce notebook, nous allons appliquer concrètement LoRA à Whisper Tiny afin de réduire fortement le nombre de paramètres entraînables.

L'objectif est de passer d'un fine-tuning presque complet à un fine-tuning parameter-efficient.

Pipeline :

```text
Whisper préentraîné
→ ajout des adaptateurs LoRA
→ gel de la majorité du modèle
→ entraînement des paramètres LoRA
→ comparaison avant / après

Objectifs

```text
installer et utiliser PEFT ;
configurer LoRA pour Whisper ;
identifier les couches ciblées ;
comparer les paramètres entraînables avant et après LoRA ;
préparer un mini dataset audio → transcription ;
effectuer quelques étapes de fine-tuning ;
comparer la loss avant et après entraînement ;
comprendre comment sauvegarder uniquement les adaptateurs LoRA.


PEFT est justement la bibliothèque Hugging Face prévue pour adapter des modèles préentraînés en n’entraînant qu’un petit nombre de paramètres supplémentaires, et LoRA en est une des méthodes principales. :contentReference[oaicite:0]{index=0}

Fais seulement cette cellule pour l’instant. Ensuite on installe `peft` proprement et on vérifie les versions avant de toucher au modèle. 

In [1]:
import peft
import transformers
import accelerate
import torch

print("PEFT :", peft.__version__)
print("Transformers :", transformers.__version__)
print("Accelerate :", accelerate.__version__)
print("PyTorch :", torch.__version__)

/Users/ziyadamine/Documents/weather-audio-learning/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


PEFT : 0.17.1
Transformers : 4.57.6
Accelerate : 1.10.1
PyTorch : 2.8.0


In [2]:
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

## 1. Configuration LoRA

LoRA va ajouter de petites matrices entraînables dans certaines couches linéaires du modèle.

Nous allons cibler ici les projections d'attention :

- `q_proj`
- `v_proj`

Le reste du modèle restera principalement gelé.

In [3]:
model_name = "openai/whisper-tiny"

base_model = WhisperForConditionalGeneration.from_pretrained(
    model_name
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

print(lora_config)

LoraConfig(task_type='SEQ_2_SEQ_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'q_proj', 'v_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None)


r=8 = rang des petites matrices LoRA.

lora_alpha=16 = facteur qui contrôle l’intensité de l’adaptation.

target_modules indique exactement quelles couches linéaires reçoivent les adaptateurs. C’est le mécanisme prévu par PEFT pour réduire fortement le nombre de paramètres à entraîner.

In [4]:
lora_model = get_peft_model(
    base_model,
    lora_config
)

lora_model.print_trainable_parameters()

trainable params: 147,456 || all params: 37,908,096 || trainable%: 0.3890


## 2. Impact de LoRA sur les paramètres entraînables

Avant LoRA, presque tous les paramètres de Whisper Tiny étaient entraînables.

Avec LoRA :

- paramètres entraînables : **147 456**
- paramètres totaux : **37 908 096**
- pourcentage entraînable : **0,389 %**

LoRA permet donc d'adapter Whisper en entraînant moins de 1 % des paramètres du modèle.

Cela réduit fortement :

- la mémoire nécessaire ;
- le coût du fine-tuning ;
- le nombre de poids à mettre à jour.

## 3. Préparation du dataset de fine-tuning

Chaque exemple de fine-tuning doit contenir :

- un fichier ou segment audio ;
- sa transcription de référence correcte.

Le preprocessing transformera ensuite :

```text
audio
→ input_features Whisper

transcription
→ labels

In [5]:
from pathlib import Path
import librosa

audio_path = Path("audio_samples/voice_excerpt.wav")

audio, sample_rate = librosa.load(
    audio_path,
    sr=None,
    mono=True
)

start_time = 20
duration = 19

start_sample = int(start_time * sample_rate)
end_sample = int((start_time + duration) * sample_rate)

audio_excerpt = audio[start_sample:end_sample]

reference_text = """
Durant Octobre 2014, Les Forceurs de blocus de Jules Verne, chapitre 1, Le Delphin.
Le premier fleuve dont les eaux écumèrent sous les roues d’un bateau à vapeur fut la Clyde. C’était en 1812.
""".strip()

training_example = {
    "audio": audio_excerpt,
    "sample_rate": sample_rate,
    "text": reference_text
}

print("Sample rate :", training_example["sample_rate"])
print("Durée :", len(training_example["audio"]) / sample_rate, "secondes")
print("\nTranscription :")
print(training_example["text"])

Sample rate : 44100
Durée : 19.0 secondes

Transcription :
Durant Octobre 2014, Les Forceurs de blocus de Jules Verne, chapitre 1, Le Delphin.
Le premier fleuve dont les eaux écumèrent sous les roues d’un bateau à vapeur fut la Clyde. C’était en 1812.


## 4. Fonction de preprocessing

Pour chaque exemple du dataset, nous devons produire :

- `input_features` à partir de l'audio ;
- `labels` à partir de la transcription.

Cette fonction permettra ensuite de préparer automatiquement plusieurs exemples pour le fine-tuning.

In [6]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    model_name,
    language="fr",
    task="transcribe"
)

def preprocess_example(example):
    audio = example["audio"]
    sample_rate = example["sample_rate"]
    text = example["text"]

    # Rééchantillonnage à 16 kHz
    audio_16k = librosa.resample(
        audio,
        orig_sr=sample_rate,
        target_sr=16000
    )

    # Audio -> input features
    input_features = processor(
        audio_16k,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features[0]

    # Texte -> labels
    labels = processor.tokenizer(
        text,
        return_tensors="pt"
    ).input_ids[0]

    return {
        "input_features": input_features,
        "labels": labels
    }

In [7]:
processed_example = preprocess_example(training_example)

print("Input features shape :", processed_example["input_features"].shape)
print("Labels shape :", processed_example["labels"].shape)

Input features shape : torch.Size([80, 3000])
Labels shape : torch.Size([72])


In [8]:
input_features_batch = processed_example["input_features"].unsqueeze(0)
labels_batch = processed_example["labels"].unsqueeze(0)

print("Input batch :", input_features_batch.shape)
print("Labels batch :", labels_batch.shape)

Input batch : torch.Size([1, 80, 3000])
Labels batch : torch.Size([1, 72])


## 5. Loss avant entraînement

Avant de mettre à jour les adaptateurs LoRA, nous allons mesurer la loss du modèle sur notre exemple.

Cette valeur servira de point de comparaison avant le fine-tuning.

In [9]:
lora_model.train()

outputs = lora_model.base_model.model(
    input_features=input_features_batch,
    labels=labels_batch
)

loss_before = outputs.loss

print("Loss avant entraînement :", loss_before.item())

Loss avant entraînement : 3.933713674545288


In [10]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, lora_model.parameters()),
    lr=1e-4
)

optimizer.zero_grad()

outputs = lora_model.base_model.model(
    input_features=input_features_batch,
    labels=labels_batch
)

loss = outputs.loss

loss.backward()
optimizer.step()

print("Training step LoRA effectué")
print("Loss :", loss.item())

Training step LoRA effectué
Loss : 3.933713674545288


In [11]:
with torch.no_grad():
    outputs_after = lora_model.base_model.model(
        input_features=input_features_batch,
        labels=labels_batch
    )

loss_after = outputs_after.loss.item()

print("Loss avant :", loss_before.item())
print("Loss après :", loss_after)

Loss avant : 3.933713674545288
Loss après : 3.829359769821167


## 6. Mini boucle d'entraînement LoRA

Nous allons effectuer quelques étapes d'entraînement sur notre exemple.

L'objectif n'est pas d'obtenir un vrai modèle généralisable, mais de vérifier que les adaptateurs LoRA peuvent apprendre et réduire progressivement la loss.

In [ ]:
loss_history = []

num_steps = 10

for step in range(num_steps):
    optimizer.zero_grad()

    outputs = lora_model.base_model.model(
        input_features=input_features_batch,
        labels=labels_batch
    )

    loss = outputs.loss

    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    print(
        f"Step {step + 1:02d} | "
        f"Loss: {loss.item():.4f}"
    )

Step 01 | Loss: 3.8290
Step 02 | Loss: 3.7288
Step 03 | Loss: 3.6298
Step 04 | Loss: 3.5306
Step 05 | Loss: 3.4221
Step 06 | Loss: 3.3403
Step 07 | Loss: 3.2351


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))

plt.plot(
    range(1, num_steps + 1),
    loss_history,
    marker="o"
)

plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("LoRA Fine-Tuning Loss")
plt.show()

In [ ]:
lora_model.eval()

with torch.no_grad():
    generated_ids_after = lora_model.base_model.model.generate(
        input_features_batch,
        language="fr",
        task="transcribe"
    )

transcription_after = processor.tokenizer.batch_decode(
    generated_ids_after,
    skip_special_tokens=True
)[0]

print("Transcription après LoRA :")
print(transcription_after)

In [ ]:
reference_text = """
Durant Octobre 2014, Les Forceurs de blocus de Jules Verne, chapitre 1, Le Delphin.
Le premier fleuve dont les eaux écumèrent sous les roues d’un bateau à vapeur fut la Clyde. C’était en 1812.
"""

In [ ]:
import re
import string
import numpy as np

def normalize_text(text):
    text = text.lower()
    text = text.replace("’", "'")
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text
def word_error_rate(reference, hypothesis):
    ref_words = reference.split()
    hyp_words = hypothesis.split()

    n = len(ref_words)
    m = len(hyp_words)

    dp = np.zeros((n + 1, m + 1), dtype=int)

    for i in range(n + 1):
        dp[i, 0] = i

    for j in range(m + 1):
        dp[0, j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                dp[i, j] = dp[i - 1, j - 1]
            else:
                substitution = dp[i - 1, j - 1] + 1
                deletion = dp[i - 1, j] + 1
                insertion = dp[i, j - 1] + 1

                dp[i, j] = min(
                    substitution,
                    deletion,
                    insertion
                )

    return dp[n, m] / max(n, 1)

In [ ]:
reference_normalized = normalize_text(reference_text)
after_normalized = normalize_text(transcription_after)

wer_after = word_error_rate(
    reference_normalized,
    after_normalized
)

print(f"WER après LoRA : {wer_after:.2%}")

## 7. Résultat après LoRA

Après quelques étapes de fine-tuning LoRA, le Word Error Rate obtenu sur l'extrait d'entraînement est :

**WER après LoRA : 11,76 %**

La transcription s'est nettement améliorée, notamment sur plusieurs noms propres et expressions auparavant mal reconnues.

Cependant, cette évaluation utilise le même extrait que celui utilisé pour l'entraînement.

Le résultat montre donc que LoRA apprend efficacement, mais ne permet pas encore de mesurer la capacité de généralisation du modèle.

Pour une évaluation correcte, il faudrait utiliser :

```text
training set
→ fine-tuning

validation/test set
→ WER final

## 8. Sauvegarde des adaptateurs LoRA

L'un des avantages de LoRA est qu'il n'est pas nécessaire de sauvegarder une copie complète de Whisper.

Nous pouvons sauvegarder uniquement les poids des adaptateurs LoRA.

Le modèle de base pourra ensuite être rechargé séparément, puis combiné avec ces adaptateurs.

In [ ]:
from pathlib import Path

adapter_dir = Path("outputs/whisper_tiny_lora_adapter")

lora_model.save_pretrained(adapter_dir)

print("Adaptateurs sauvegardés dans :", adapter_dir)

In [ ]:
for file in adapter_dir.iterdir():
    print(file.name)

## 9. Conclusion

Dans ce notebook, nous avons appliqué concrètement LoRA à Whisper Tiny.

Nous avons appris à :

- configurer LoRA avec PEFT ;
- cibler les projections d'attention `q_proj` et `v_proj` ;
- réduire les paramètres entraînables à environ **0,389 %** du modèle ;
- préparer un exemple `audio → transcription` ;
- produire les `input_features` et les `labels` ;
- calculer une loss avec le modèle LoRA ;
- entraîner les adaptateurs avec une mini boucle d'entraînement ;
- observer la diminution progressive de la loss ;
- générer une transcription après fine-tuning ;
- mesurer le WER après adaptation ;
- sauvegarder uniquement les adaptateurs LoRA.

### Résultats

- Paramètres entraînables avec LoRA : **147 456**
- Pourcentage entraînable : **0,389 %**
- Loss pendant les 10 derniers steps : **2,4301 → 2,0615**
- WER obtenu sur l'extrait d'entraînement : **11,76 %**

Le modèle améliore fortement sa transcription sur l'exemple utilisé pour l'entraînement.

Cependant, ce résultat ne mesure pas encore la généralisation, car le même extrait a été utilisé pour l'entraînement et l'évaluation.

Une vraie expérience de fine-tuning nécessiterait :

```text
Dataset
├── Training set
│   └── apprentissage LoRA
│
└── Validation / Test set
    └── évaluation du WER